In [18]:
# External Imports
import sys
from pathlib import Path
import glob
import matplotlib.pyplot as plt
import torch
from torchinfo import summary

# Internal Imports
sys.path.insert(0, '../src')
from src.DataIntegrity import data_integrity_check
from src.Dataset import split_patients, MriDataset, get_train_transforms, get_val_transforms, get_dataloaders
from src.Model import BaselineModel

In [2]:
data_path = Path("../data/raw/lgg-mri-segmentation/kaggle_3m")
rejected_path = Path("../data/processed/data-integrity")

In [3]:
accepted_data, rejected_data = data_integrity_check(data_path)

[INFO] Data Integrity Checks: 100%|██████████| 112/112 [00:10<00:00, 10.23it/s]

[Info] Data Integrity Pipeline finished
[Info] Segments Accepted = 3929 | Segments Rejected = 0


In [7]:
SPLIT_SEED = 42
train_patients, val_patients, test_patients = split_patients(accepted_data, seed=SPLIT_SEED)

[INFO]  Splitting dataset with seed 42...
[INFO]  Train Set: 76  | Tumor Ratio: 0.352
[INFO]  Valid Set: 17  | Tumor Ratio: 0.360
[INFO]  Test Set:  17  | Tumor Ratio: 0.372


In [12]:
BATCH_SIZE = 2
train_dataloader, val_dataloader, test_dataloader = get_dataloaders(train_patients, val_patients, test_patients, batch_size=BATCH_SIZE)
images, labels = next(iter(train_dataloader))

In [10]:
baseline_model = BaselineModel()

In [15]:
output = baseline_model(images)

In [17]:
output.shape

torch.Size([2, 2])

In [20]:
summary(baseline_model, input_size=(BATCH_SIZE, 3, 224, 224))

Layer (type:depth-idx)                   Output Shape              Param #
BaselineModel                            [2, 2]                    --
├─Sequential: 1-1                        [2, 32, 111, 111]         --
│    └─Conv2d: 2-1                       [2, 32, 223, 223]         416
│    └─ReLU: 2-2                         [2, 32, 223, 223]         --
│    └─MaxPool2d: 2-3                    [2, 32, 111, 111]         --
├─Sequential: 1-2                        [2, 32, 55, 55]           --
│    └─Conv2d: 2-4                       [2, 32, 110, 110]         4,128
│    └─ReLU: 2-5                         [2, 32, 110, 110]         --
│    └─MaxPool2d: 2-6                    [2, 32, 55, 55]           --
├─Flatten: 1-3                           [2, 96800]                --
├─Sequential: 1-4                        [2, 256]                  --
│    └─Linear: 2-7                       [2, 256]                  24,781,056
│    └─ReLU: 2-8                         [2, 256]                  --
├─L